# 16 · Análisis de datos con pandas

**pandas** es la biblioteca más usada para analizar datos en tablas. Sus dos estructuras principales:

- **`Series`**: una columna (como una lista con etiquetas).
- **`DataFrame`**: una tabla de varias columnas.

Se importa por convención como `pd`.

In [ ]:
import pandas as pd

## Crear un DataFrame

Desde un diccionario de listas (cada llave = una columna):

In [ ]:
df = pd.DataFrame({
    "producto": ["Lápiz", "Cuaderno", "Borrador", "Regla", "Mochila", "Lápiz"],
    "categoria": ["Escritura", "Papelería", "Escritura", "Papelería", "Accesorios", "Escritura"],
    "precio": [5.0, 25.0, 3.5, 12.0, 150.0, 5.0],
    "cantidad": [30, 10, 25, 8, 2, 15],
})
df

Una **lista de diccionarios** (cuaderno 12) también funciona:

In [ ]:
pd.DataFrame([{"a": 1, "b": 2}, {"a": 3, "b": 4}])

## Explorar los datos

In [ ]:
print(df.shape)       # (filas, columnas)
print(df.columns.tolist())
df.head(3)            # primeras 3 filas (tail() = últimas)

In [ ]:
df.info()

In [ ]:
df.describe()         # estadísticas de las columnas numéricas

## Seleccionar columnas y filas

In [ ]:
print(df["producto"])           # una columna -> Series
df[["producto", "precio"]]      # varias columnas -> DataFrame

In [ ]:
df.iloc[0]          # fila por posición

In [ ]:
df.iloc[1:3]        # filas 1 y 2

In [ ]:
df.loc[0, "producto"]   # por etiqueta de fila y nombre de columna

## Filtrar filas

Se usa una condición; se pueden combinar con `&` (y), `|` (o), `~` (no), **con paréntesis**.

In [ ]:
df[df["precio"] > 10]

In [ ]:
df[(df["categoria"] == "Escritura") & (df["cantidad"] >= 20)]

In [ ]:
df[df["producto"].isin(["Lápiz", "Regla"])]

## Crear columnas nuevas

In [ ]:
df["total"] = df["precio"] * df["cantidad"]
df

## Estadísticas

In [ ]:
print(df["precio"].mean())     # promedio
print(df["precio"].max(), df["precio"].min())
print(df["total"].sum())
print(df["producto"].nunique())          # valores distintos
print(df["categoria"].value_counts())    # frecuencia de cada valor

## Ordenar

In [ ]:
df.sort_values("total", ascending=False)

## Agrupar: `groupby`

Divide por categorías y calcula algo para cada grupo.

In [ ]:
df.groupby("categoria")["total"].sum()

In [ ]:
df.groupby("categoria").agg(
    productos=("producto", "count"),
    precio_promedio=("precio", "mean"),
    ingreso_total=("total", "sum"),
)

## Valores faltantes

In [ ]:
import numpy as np

datos = pd.DataFrame({"x": [1, 2, np.nan, 4], "y": ["a", None, "c", "d"]})
print(datos.isna().sum())     # cuántos faltan por columna
print(datos.fillna(0))        # rellenar
print(datos.dropna())         # eliminar filas incompletas

## Leer y escribir archivos CSV

In [ ]:
df.to_csv("inventario.csv", index=False)
leido = pd.read_csv("inventario.csv")
leido.head()

`pd.read_csv("archivo.csv")` es la forma habitual de cargar datos propios. También existen `read_excel`, `read_json`, etc.

## Un conjunto más grande

Generamos 90 ventas aleatorias (con semilla, para que todos obtengan lo mismo).

In [ ]:
import random
random.seed(1)

ventas = pd.DataFrame({
    "mes": random.choices(["Ene", "Feb", "Mar"], k=90),
    "tienda": random.choices(["Norte", "Centro", "Sur"], k=90),
    "producto": random.choices(["Lápiz", "Cuaderno", "Mochila"], k=90),
    "unidades": [random.randint(1, 20) for _ in range(90)],
})
precios = {"Lápiz": 5, "Cuaderno": 25, "Mochila": 150}
ventas["ingreso"] = ventas["unidades"] * ventas["producto"].map(precios)
ventas.head()

In [ ]:
ventas.to_csv("ventas.csv", index=False)   # lo reutilizaremos en el cuaderno 17
ventas.groupby(["tienda", "producto"])["ingreso"].sum().unstack()

## 🎛️ Filtros interactivos

In [ ]:
import ipywidgets as widgets
from IPython.display import display

tienda = widgets.Dropdown(options=["Todas"] + sorted(ventas["tienda"].unique()), description="Tienda:")
producto = widgets.Dropdown(options=["Todos"] + sorted(ventas["producto"].unique()), description="Producto:")
min_u = widgets.IntSlider(value=1, min=1, max=20, description="Unid. ≥")
out = widgets.Output()

def filtrar(_=None):
    sub = ventas[ventas["unidades"] >= min_u.value]
    if tienda.value != "Todas":
        sub = sub[sub["tienda"] == tienda.value]
    if producto.value != "Todos":
        sub = sub[sub["producto"] == producto.value]
    with out:
        out.clear_output(wait=True)
        print(f"{len(sub)} filas · ingreso total: {sub['ingreso'].sum():,}")
        display(sub.head(8))

for w in (tienda, producto, min_u):
    w.observe(filtrar, names="value")
display(tienda, producto, min_u, out)
filtrar()

## Resumen

| Tarea | Código |
|-------|--------|
| Crear | `pd.DataFrame({...})` |
| Explorar | `.head()`, `.info()`, `.describe()`, `.shape` |
| Columna | `df["col"]` |
| Filtrar | `df[df["col"] > 10]` |
| Nueva columna | `df["nueva"] = ...` |
| Agrupar | `df.groupby("col")["otra"].sum()` |
| Ordenar | `df.sort_values("col")` |
| Leer / guardar | `pd.read_csv()`, `df.to_csv()` |